# 02 — Narrative Inspection

Load generated narratives from `outputs/generation/<run_id>/narratives.csv`, read them
alongside their ground-truth SHAP values, and manually inspect quality.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from dotenv import load_dotenv

load_dotenv('../.env')

from src.config import load_config
from src.storage import list_runs, load_narratives_csv, narratives_csv_path, run_dir

cfg = load_config('../config/default.yaml')

In [ ]:
# List all available runs (folders with narratives.csv)
runs = list_runs(f'../{cfg.storage.generation_dir}')
pd.DataFrame(runs)

In [ ]:
# Set the run_id you want to inspect
RUN_ID = runs[0]['run_id'] if runs else None
print('Inspecting run:', RUN_ID)

In [ ]:
csv_path = narratives_csv_path(run_dir(cfg, RUN_ID))
narr_df = load_narratives_csv(csv_path)
print(f'{len(narr_df)} narratives loaded from {csv_path}')
narr_df.head()

In [ ]:
# Summary: narratives per model × dataset
narr_df.groupby(['model_id', 'dataset']).size().unstack(fill_value=0)

In [ ]:
# Inspect a random narrative alongside its SHAP context
import pandas as pd as pd2  # avoid re-import warning
from src.data_loader import format_shap_table

sample = narr_df.sample(1).iloc[0]
dataset_cfg = cfg.get_dataset(sample['dataset'])
raw_df = pd.read_csv(f'../{dataset_cfg.path}')
row = raw_df.iloc[sample['instance_id']]

print('=== SHAP VALUES ===')
print(format_shap_table(row, dataset_cfg.shap_col_prefix))
print()
print(f'=== NARRATIVE ({sample["model_id"]} | {sample["dataset"]}) ===')
print(sample['narrative_text'])

In [ ]:
# Browse narratives for a specific model and dataset
MODEL = 'claude-opus'
DATASET = 'adult'
N = 5  # how many to show

subset = narr_df[
    (narr_df['model_id'] == MODEL) &
    (narr_df['dataset'] == DATASET) &
    (narr_df['error'].fillna('') == '')
].head(N)

for _, row in subset.iterrows():
    print(f'--- Instance {row["instance_id"]} ---')
    print(row['narrative_text'])
    print()